In [1]:
# ============================================================
# CELL 1: SETUP & IMPORTS (with CUDA fallback)
# ============================================================

import os, re, cv2, copy, time, random, gc, math, hashlib, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from collections import Counter, defaultdict
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, classification_report,
    f1_score, cohen_kappa_score
)
from scipy.optimize import minimize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

# ── CUDA compatibility check and fallback ──
def get_device():
    """Get device with CUDA fallback to CPU."""
    if torch.cuda.is_available():
        try:
            # Test CUDA with a simple operation
            test_tensor = torch.zeros(1).cuda()
            del test_tensor
            torch.cuda.empty_cache()
            cuda_available = True
        except RuntimeError as e:
            print(f"⚠ CUDA error detected: {e}")
            print("  Falling back to CPU (this may be slow)")
            cuda_available = False
    else:
        cuda_available = False
    
    return torch.device("cuda" if cuda_available else "cpu"), cuda_available

DEVICE, CUDA_AVAILABLE = get_device()
USE_AMP = CUDA_AVAILABLE  # Only use AMP if CUDA is available

SEED = 42

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): 
        try:
            torch.cuda.manual_seed_all(s)
        except: pass
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
if CUDA_AVAILABLE:
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    except: pass

print(f"Device: {DEVICE} | AMP: {USE_AMP}")
if CUDA_AVAILABLE:
    try:
        print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
    except: 
        print("GPU info unavailable")
else:
    print("⚠ Using CPU — inference will be slower")

Device: cuda | AMP: True
GPU: Tesla T4 | VRAM: 15.6GB


In [2]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================

class V6CFG:
    """Configuration for V6 model (DINOv2-B + DenseNet121 = 1792d)"""
    model_dir     = "/kaggle/input/datasets/ronakp004/bladder-research-ensemble/saved_models/saved_models/v6"
    feat_dim      = 1792       # DINOv2-B(768) + DenseNet121(1024)
    dino_model_name = 'dinov2_vitb14'
    dino_dim      = 768
    dense_dim     = 1024
    mil_hidden    = 512
    mil_dropout   = 0.25
    clam_k_sample = 10
    n_att_heads   = 4
    feat_noise_std = 0.02
    feat_drop_p   = 0.1
    num_classes   = 3
    n_ensemble    = 3
    max_patches_test = 400
    use_mc_dropout = True
    use_tta       = True
    tta_rounds    = 3

class V9CFG:
    """Configuration for V9 model (DINOv2-L(ft) + DenseNet121 = 2048d)"""
    weights_dir   = "/kaggle/input/datasets/ronakp004/bladder-research-ensemble/Weights/Weights"
    primary_dim   = 1024       # DINOv2-L
    frozen_dim    = 1024       # DenseNet121
    total_dim     = 2048
    dino_model_name = 'dinov2_vitl14'
    mil_hidden    = 384
    mil_dropout   = 0.25
    clam_k        = 8
    noise_std     = 0.02
    feat_drop     = 0.1
    n_heads       = 4
    num_classes   = 3
    n_ensemble    = 3
    max_test_patches = 300
    tta_rounds    = 3
    mc_forward    = 5
    frozen_batch  = 64
    max_patches   = 50

class CFG:
    """Shared configuration"""
    data_root     = "/kaggle/input/datasets/ronakp004/bladder-research-ensemble/EndoscopicBladderTissue/EndoscopicBladderTissue"
    aug_root      = "/kaggle/input/datasets/ronakp004/bladder-research-ensemble/Augmented Data/Augmented Data"
    aug_manifest  = "/kaggle/input/datasets/ronakp004/bladder-research-ensemble/Augmented Data/Augmented Data/augmented_only_manifest.csv"
    class_names   = ['HGC', 'LGC', 'Normal']
    num_classes   = 3
    image_resize  = 512
    clahe_clip    = 1.5
    clahe_grid    = (16, 16)
    patch_scales  = [96, 128, 192]
    patch_output  = 224
    stride_frac   = 0.5
    min_tissue    = 0.40
    max_bright    = 245
    min_bright    = 15
    min_sat       = 10
    min_focus     = 8.0
    quality_frac  = 0.85
    max_patches   = 60         # V6 default
    results_dir   = "ensemble_results"

# Dynamic device-aware normalization tensors
def get_normalization_tensors():
    """Get ImageNet normalization tensors on the correct device."""
    mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(DEVICE)
    std = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(DEVICE)
    return mean, std

IMNET_MEAN, IMNET_STD = get_normalization_tensors()
LABEL_MAP  = {'HGC':'HGC','LGC':'LGC','NST':'Normal','NTL':'Normal'}

os.makedirs(CFG.results_dir, exist_ok=True)
print("✓ Configuration loaded (V6 + V9 + shared)")
print(f"  V6: {V6CFG.feat_dim}d ({V6CFG.dino_model_name} + DenseNet121)")
print(f"  V9: {V9CFG.total_dim}d ({V9CFG.dino_model_name}[ft] + DenseNet121)")
print(f"  Normalization tensors on device: {IMNET_MEAN.device}")

✓ Configuration loaded (V6 + V9 + shared)
  V6: 1792d (dinov2_vitb14 + DenseNet121)
  V9: 2048d (dinov2_vitl14[ft] + DenseNet121)
  Normalization tensors on device: cuda:0


In [3]:
# ============================================================
# CELL 3: LOAD DATASETS
# ============================================================

print("\n" + "="*60)
print("LOADING DATASETS")
print("="*60)

records = []; pat_re = re.compile(r'pt[_]?0*(\d+)')
for lbl in os.listdir(CFG.data_root):
    lp = os.path.join(CFG.data_root, lbl)
    if not os.path.isdir(lp) or lbl not in LABEL_MAP: continue
    ml = LABEL_MAP[lbl]
    for fn in os.listdir(lp):
        if not fn.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tif')): continue
        m = pat_re.search(fn)
        if m:
            records.append({"path":os.path.join(lp,fn),"label":ml,
                "original_label":lbl,"patient":int(m.group(1)),"filename":fn,
                "is_augmented":False,"aug_mode":"none"})

df = pd.DataFrame(records)
class_to_idx = {c:i for i,c in enumerate(CFG.class_names)}
idx_to_class = {i:c for c,i in class_to_idx.items()}
df["target"] = df["label"].map(class_to_idx)
PATIENTS = sorted(df.patient.unique())
N_PATIENTS = len(PATIENTS)

cc = df['label'].value_counts(); tot = len(df)
print(f"✓ {len(df)} original images, {N_PATIENTS} patients")
for c in CFG.class_names:
    print(f"  {c}: {cc.get(c,0)} ({cc.get(c,0)/tot:.1%})")

# Patient-to-fold mapping (1-indexed)
patient_to_fold = {pid: fold_idx+1 for fold_idx, pid in enumerate(PATIENTS)}
print(f"\nPatient → Fold mapping:")
for pid, fold in patient_to_fold.items():
    print(f"  P{pid} → Fold {fold:02d}")


LOADING DATASETS
✓ 1713 original images, 14 patients
  HGC: 469 (27.4%)
  LGC: 647 (37.8%)
  Normal: 597 (34.9%)

Patient → Fold mapping:
  P1 → Fold 01
  P2 → Fold 02
  P3 → Fold 03
  P4 → Fold 04
  P5 → Fold 05
  P6 → Fold 06
  P7 → Fold 07
  P8 → Fold 08
  P9 → Fold 09
  P10 → Fold 10
  P11 → Fold 11
  P12 → Fold 12
  P13 → Fold 13
  P14 → Fold 14


In [4]:
# ============================================================
# CELL 4: PREPROCESSING UTILITIES
# ============================================================

class LabNormalizer:
    def __init__(self): self.ref = None
    def fit(self, imgs):
        st = {'L':[],'a':[],'b':[]}
        for img in imgs:
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            for i,ch in enumerate(['L','a','b']):
                st[ch].append({'m':lab[:,:,i].mean(),'s':lab[:,:,i].std()+1e-6})
        self.ref = {ch:{'m':np.median([x['m'] for x in st[ch]]),
                        's':np.median([x['s'] for x in st[ch]])} for ch in ['L','a','b']}
        return self
    def transform(self, img):
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
        for i,ch in enumerate(['L','a','b']):
            c = lab[:,:,i]; sm,ss = c.mean(), c.std()+1e-6
            lab[:,:,i] = np.clip((c-sm)*(self.ref[ch]['s']/ss)+self.ref[ch]['m'],0,255)
        lab = lab.astype(np.uint8)
        lab[:,:,0] = cv2.createCLAHE(clipLimit=CFG.clahe_clip,
                                      tileGridSize=CFG.clahe_grid).apply(lab[:,:,0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def load_image(path, norm=None):
    img = cv2.imread(path)
    if img is None: raise FileNotFoundError(path)
    h,w = img.shape[:2]; s = CFG.image_resize/max(h,w)
    if s!=1: img = cv2.resize(img,(int(w*s),int(h*s)),interpolation=cv2.INTER_AREA)
    return norm.transform(img) if norm else img

def patch_quality(p):
    hsv = cv2.cvtColor(p, cv2.COLOR_BGR2HSV)
    v,s = hsv[:,:,2].astype(np.float32), hsv[:,:,1].astype(np.float32)
    mask = (v<CFG.max_bright)&(v>CFG.min_bright)&(s>CFG.min_sat)
    tf = mask.sum()/mask.size
    if tf < CFG.min_tissue: return -1.0
    gray = cv2.cvtColor(p, cv2.COLOR_BGR2GRAY)
    foc = cv2.Laplacian(gray, cv2.CV_64F).var()
    if foc < CFG.min_focus: return -1.0
    ss = s[mask].std()/50 if mask.sum()>10 else 0
    ed = cv2.Canny(gray,50,150).sum()/(255*gray.size)*10
    return 0.3*tf+0.3*min(foc/100,1)+0.2*min(ss,1)+0.2*min(ed,1)

def extract_patches(img, maxp=None):
    if maxp is None: maxp = CFG.max_patches
    H,W = img.shape[:2]; cands = []; cap = maxp*3
    for sc in CFG.patch_scales:
        if sc>min(H,W): continue
        st = max(1,int(sc*CFG.stride_frac))
        for y in range(0,H-sc+1,st):
            for x in range(0,W-sc+1,st):
                if len(cands)>=cap: break
                cr = img[y:y+sc,x:x+sc]; q = patch_quality(cr)
                if q>0: cands.append((cv2.resize(cr,(CFG.patch_output,CFG.patch_output),
                                      interpolation=cv2.INTER_AREA), q))
            if len(cands)>=cap: break
        if len(cands)>=cap: break
    if not cands: return []
    cands.sort(key=lambda x:x[1], reverse=True)
    nk = max(1,int(len(cands)*CFG.quality_frac))
    return [c[0] for c in cands[:nk][:maxp]]

def bgr_to_tensor(p):
    return torch.from_numpy(cv2.cvtColor(p,cv2.COLOR_BGR2RGB)).permute(2,0,1).float()/255.0

# Fit global normalizer
samps = []
for pid in PATIENTS:
    for fp in df[df.patient==pid].path.values[:12]:
        try:
            img = cv2.imread(fp)
            if img is not None:
                h,w = img.shape[:2]; s = CFG.image_resize/max(h,w)
                if s!=1: img = cv2.resize(img,(int(w*s),int(h*s)))
                samps.append(img)
        except: pass
        if len(samps)>=200: break
    if len(samps)>=200: break
normalizer = LabNormalizer().fit(samps); del samps
print("✓ Preprocessing utilities ready + normalizer fitted")

✓ Preprocessing utilities ready + normalizer fitted


In [5]:
# ============================================================
# CELL 5: V6 CLAM MODEL DEFINITION
# ============================================================

class V6MultiHeadGatedAttention(nn.Module):
    def __init__(self, hidden, n_heads=4):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = hidden // 2 // n_heads
        self.att_nets = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, self.head_dim), nn.Tanh())
            for _ in range(n_heads)
        ])
        self.gate_nets = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, self.head_dim), nn.Sigmoid())
            for _ in range(n_heads)
        ])
        self.combine = nn.Linear(self.head_dim * n_heads, hidden // 2)

    def forward(self, h):
        heads = [a(h) * g(h) for a, g in zip(self.att_nets, self.gate_nets)]
        return self.combine(torch.cat(heads, dim=-1))


class V6CLAM(nn.Module):
    def __init__(self, feat_dim=V6CFG.feat_dim, hidden=V6CFG.mil_hidden,
                 n_classes=V6CFG.num_classes, dropout=V6CFG.mil_dropout,
                 k_sample=V6CFG.clam_k_sample, n_heads=V6CFG.n_att_heads):
        super().__init__()
        self.n_classes  = n_classes
        self.k_sample   = k_sample
        self.feat_noise = V6CFG.feat_noise_std
        self.feat_drop  = nn.Dropout(V6CFG.feat_drop_p)

        self.adapter = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.LayerNorm(feat_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(feat_dim, feat_dim),
        )
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.mh_attention = V6MultiHeadGatedAttention(hidden, n_heads)
        self.att_temp = nn.Parameter(torch.ones(n_classes))

        self.att_branches = nn.ModuleList([
            nn.Linear(hidden // 2, 1) for _ in range(n_classes)
        ])
        self.inst_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, 128), nn.GELU(),
                nn.Dropout(0.1), nn.Linear(128, 2)
            ) for _ in range(n_classes)
        ])
        self.bag_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden // 4),
                nn.GELU(),
                nn.Linear(hidden // 4, 1)
            ) for _ in range(n_classes)
        ])

    def forward(self, x, label=None):
        x = x.float()
        if self.training:
            x = x + torch.randn_like(x) * self.feat_noise
            x = self.feat_drop(x)
        x = x + self.adapter(x)
        h = self.fc(x)
        att = self.mh_attention(h)
        logits = []
        total_inst = torch.tensor(0.0, device=x.device)
        for c in range(self.n_classes):
            a_scores = self.att_branches[c](att).squeeze(-1)
            a_scores = a_scores / (self.att_temp[c].abs() + 0.1)
            a_weights = F.softmax(a_scores, dim=0)
            bag = torch.sum(a_weights.unsqueeze(-1) * h, dim=0)
            logits.append(self.bag_classifiers[c](bag))
        return {'logits': torch.cat(logits), 'inst_loss': total_inst}

print(f"✓ V6 CLAM defined: {V6CFG.feat_dim}d → {V6CFG.mil_hidden}h → {V6CFG.num_classes} classes")

✓ V6 CLAM defined: 1792d → 512h → 3 classes


In [6]:
# ============================================================
# CELL 6: V9 CLAM MODEL DEFINITION
# ============================================================

class V9MultiHeadGatedAttn(nn.Module):
    def __init__(self, dim, n_heads=V9CFG.n_heads, drop=0.25):
        super().__init__()
        self.n_heads = n_heads
        hd = max(dim // n_heads, 32)
        self.att = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, hd), nn.Tanh()) for _ in range(n_heads)])
        self.gate = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, hd), nn.Sigmoid()) for _ in range(n_heads)])
        self.score = nn.ModuleList([nn.Linear(hd, 1) for _ in range(n_heads)])
        self.drop = nn.Dropout(drop)

    def forward(self, h):
        scores = []
        for i in range(self.n_heads):
            gated = self.att[i](h) * self.gate[i](h)
            s = self.score[i](self.drop(gated))
            scores.append(s)
        A = torch.cat(scores, dim=-1).mean(dim=-1, keepdim=True)
        A = F.softmax(A, dim=0)
        return A


class V9CLAM(nn.Module):
    def __init__(self, fdim=None, nc=V9CFG.num_classes, hid=V9CFG.mil_hidden,
                 drop=V9CFG.mil_dropout, k=V9CFG.clam_k):
        super().__init__()
        if fdim is None: fdim = V9CFG.total_dim
        self.k = k; self.nc = nc
        self.enc = nn.Sequential(
            nn.Linear(fdim, hid),
            nn.LayerNorm(hid),
            nn.GELU(),
            nn.Dropout(drop),
        )
        self.attn = V9MultiHeadGatedAttn(hid, drop=drop)
        self.clf = nn.Sequential(
            nn.Linear(hid, hid // 2),
            nn.GELU(),
            nn.Dropout(drop * 0.5),
            nn.Linear(hid // 2, nc),
        )
        self.inst_clf = nn.Linear(hid, nc)

    def forward(self, x, ret_attn=False):
        if self.training:
            if V9CFG.noise_std > 0:
                x = x + torch.randn_like(x) * V9CFG.noise_std
            if V9CFG.feat_drop > 0:
                x = x * (torch.rand(x.shape[0], 1, device=x.device) > V9CFG.feat_drop).float()
        h = self.enc(x)
        A = self.attn(h)
        bag = (A * h).sum(dim=0, keepdim=True)
        logits = self.clf(bag)
        inst = self.inst_clf(h) if self.training and self.k > 0 else None
        if ret_attn:
            return logits, inst, A.detach()
        return logits, inst

print(f"✓ V9 CLAM defined: {V9CFG.total_dim}d → {V9CFG.mil_hidden}h → {V9CFG.num_classes} classes")

✓ V9 CLAM defined: 2048d → 384h → 3 classes


In [7]:
# ============================================================
# CELL 7: BACKBONE LOADING + FEATURE EXTRACTION
# ============================================================

def get_dino_feat(model, x):
    """Safely extract CLS token from DINOv2 output."""
    out = model(x)
    if isinstance(out, dict):
        return out.get('x_norm_clstoken', next(iter(out.values())))
    if out.dim() > 2: return out[:,0,:]
    return out

def load_dino_v6():
    """Load DINOv2-B for V6 (frozen)."""
    print("  Loading DINOv2-B for V6...")
    try:
        model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        model = model.to(DEVICE).eval()
    except RuntimeError as e:
        if 'CUDA' in str(e) or 'cuda' in str(e):
            print(f"  ⚠ CUDA error loading V6 model, using CPU: {e}")
            model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
            model = model.to('cpu').eval()
        else:
            raise
    for p in model.parameters(): p.requires_grad = False
    print(f"  ✓ dinov2_vitb14 — {V6CFG.dino_dim}d on {next(model.parameters()).device}")
    return model

def load_dino_v9():
    """Load DINOv2-L for V9 (will load per-fold fine-tuned weights)."""
    print("  Loading DINOv2-L for V9...")
    try:
        model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
        model = model.to(DEVICE).eval()
    except RuntimeError as e:
        if 'CUDA' in str(e) or 'cuda' in str(e):
            print(f"  ⚠ CUDA error loading V9 model, using CPU: {e}")
            model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
            model = model.to('cpu').eval()
        else:
            raise
    print(f"  ✓ dinov2_vitl14 — {V9CFG.primary_dim}d on {next(model.parameters()).device}")
    return model

def load_densenet():
    """Load DenseNet121 (shared frozen backbone)."""
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    feat_dim = model.classifier.in_features
    model.classifier = nn.Identity()
    model.eval()
    for p in model.parameters(): p.requires_grad = False
    try:
        model = model.to(DEVICE)
    except RuntimeError as e:
        if 'CUDA' in str(e) or 'cuda' in str(e):
            print(f"  ⚠ CUDA error loading DenseNet, using CPU: {e}")
            model = model.to('cpu')
        else:
            raise
    print(f"  ✓ DenseNet121 — {feat_dim}d (frozen) on {next(model.parameters()).device}")
    return model, feat_dim

print("Loading backbones...")
dino_v6_model = load_dino_v6()
dino_v9_model = load_dino_v9()
# Save original V9 weights for per-fold reset
original_v9_sd = copy.deepcopy(dino_v9_model.state_dict())
dense_model, dense_dim = load_densenet()
print(f"\n✓ All backbones loaded")

Loading backbones...
  Loading DINOv2-B for V6...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:00<00:00, 359MB/s] 


  ✓ dinov2_vitb14 — 768d on cuda:0
  Loading DINOv2-L for V9...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:03<00:00, 377MB/s]


  ✓ dinov2_vitl14 — 1024d on cuda:0
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 130MB/s] 


  ✓ DenseNet121 — 1024d (frozen) on cuda:0

✓ All backbones loaded


In [8]:
# ============================================================
# CELL 8: FEATURE EXTRACTION FUNCTIONS (with CUDA error handling)
# ============================================================

@torch.inference_mode()
def extract_v6_features(patch_tensors):
    """Extract dual-backbone features for V6 (DINOv2-B + DenseNet = 1792d)."""
    if not patch_tensors:
        return torch.empty((0, V6CFG.feat_dim), dtype=torch.float16)
    all_feats = []
    batch_size = 128
    for i in range(0, len(patch_tensors), batch_size):
        try:
            batch = torch.stack(patch_tensors[i:i+batch_size]).to(DEVICE, non_blocking=True)
            batch_norm = (batch - IMNET_MEAN) / IMNET_STD
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                dino_out = get_dino_feat(dino_v6_model, batch_norm)
                dense_out = dense_model(batch_norm)
            combined = torch.cat([dino_out.float().cpu(), dense_out.float().cpu()], dim=1)
            all_feats.append(combined)
        except RuntimeError as e:
            if 'CUDA' in str(e) or 'cuda' in str(e):
                print(f"⚠ CUDA error in v6 extraction: {e}")
                print("  Switching to CPU for this batch")
                batch = torch.stack(patch_tensors[i:i+batch_size]).to('cpu')
                batch_norm = (batch - IMNET_MEAN.cpu()) / IMNET_STD.cpu()
                dino_out = get_dino_feat(dino_v6_model, batch_norm)
                dense_out = dense_model(batch_norm)
                combined = torch.cat([dino_out.float(), dense_out.float()], dim=1)
                all_feats.append(combined)
            else:
                raise
    if not all_feats:
        return torch.empty((0, V6CFG.feat_dim), dtype=torch.float16)
    return torch.cat(all_feats, 0).half()

@torch.inference_mode()
def extract_v9_features(patch_tensors, primary_backbone):
    """Extract dual-backbone features for V9 (fine-tuned DINOv2-L + DenseNet = 2048d)."""
    if not patch_tensors:
        return torch.empty((0, V9CFG.total_dim), dtype=torch.float16)
    all_feats = []
    batch_size = V9CFG.frozen_batch
    for i in range(0, len(patch_tensors), batch_size):
        try:
            batch = torch.stack(patch_tensors[i:i+batch_size]).to(DEVICE, non_blocking=True)
            batch_norm = (batch - IMNET_MEAN) / IMNET_STD
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                primary_out = get_dino_feat(primary_backbone, batch_norm)
                dense_out = dense_model(batch_norm)
            combined = torch.cat([primary_out.float().cpu(), dense_out.float().cpu()], dim=1)
            all_feats.append(combined)
        except RuntimeError as e:
            if 'CUDA' in str(e) or 'cuda' in str(e):
                print(f"⚠ CUDA error in v9 extraction: {e}")
                print("  Switching to CPU for this batch")
                batch = torch.stack(patch_tensors[i:i+batch_size]).to('cpu')
                batch_norm = (batch - IMNET_MEAN.cpu()) / IMNET_STD.cpu()
                primary_out = get_dino_feat(primary_backbone, batch_norm)
                dense_out = dense_model(batch_norm)
                combined = torch.cat([primary_out.float(), dense_out.float()], dim=1)
                all_feats.append(combined)
            else:
                raise
    if not all_feats:
        return torch.empty((0, V9CFG.total_dim), dtype=torch.float16)
    return torch.cat(all_feats, 0).half()

print("✓ Feature extraction functions defined (V6: 1792d, V9: 2048d, with CUDA fallback)")

✓ Feature extraction functions defined (V6: 1792d, V9: 2048d, with CUDA fallback)


In [9]:
# ============================================================
# CELL 9: WEIGHT LOADING + PREDICTION FUNCTIONS
# ============================================================

def load_v6_models_for_fold(fold_idx):
    """Load 3 V6 CLAM models for a given fold (1-indexed)."""
    models_list = []
    for seed in range(1, V6CFG.n_ensemble + 1):
        path = os.path.join(V6CFG.model_dir, f"fold_{fold_idx:02d}_seed_{seed}.pth")
        if not os.path.exists(path):
            print(f"  ⚠ V6 weight not found: {path}")
            continue
        try:
            ckpt = torch.load(path, map_location='cpu', weights_only=False)
            model = V6CLAM().to(DEVICE)
            model.load_state_dict(ckpt['state_dict'])
            model.eval()
            models_list.append(model)
        except RuntimeError as e:
            if 'CUDA' in str(e) or 'cuda' in str(e):
                print(f"  ⚠ CUDA error loading V6 model, using CPU: {path}")
                ckpt = torch.load(path, map_location='cpu', weights_only=False)
                model = V6CLAM().to('cpu')
                model.load_state_dict(ckpt['state_dict'])
                model.eval()
                models_list.append(model)
            else:
                raise
    return models_list

def load_v9_backbone_for_fold(patient_id):
    """Load fine-tuned V9 backbone for a given patient fold."""
    fold_dir = os.path.join(V9CFG.weights_dir, f"P{patient_id}")
    bb_path = os.path.join(fold_dir, "primary_backbone.pt")
    if not os.path.exists(bb_path):
        print(f"  ⚠ V9 backbone not found: {bb_path}")
        return None
    # Validate file — detect corrupt downloads (XML error pages, expired tokens)
    try:
        with open(bb_path, 'rb') as f:
            header = f.read(16)
        if header[:5] == b'<?xml' or header[:1] == b'<':
            size_mb = os.path.getsize(bb_path) / 1e6
            print(f"  ⚠ V9 backbone CORRUPT (XML error page, {size_mb:.1f}MB): P{patient_id}")
            print(f"    → V9 will be skipped, using V6 only for this fold")
            return None
        ckpt = torch.load(bb_path, map_location='cpu', weights_only=False)
        try:
            dino_v9_model.load_state_dict(ckpt['state_dict'])
            dino_v9_model.to(DEVICE).eval()
        except RuntimeError as e:
            if 'CUDA' in str(e) or 'cuda' in str(e):
                print(f"  ⚠ CUDA error loading V9 backbone, using CPU: {e}")
                dino_v9_model.to('cpu').eval()
            else:
                raise
        return dino_v9_model
    except Exception as e:
        print(f"  ⚠ V9 backbone load failed for P{patient_id}: {e}")
        return None

def load_v9_clam_for_fold(patient_id):
    """Load 3 V9 CLAM models for a given patient fold."""
    fold_dir = os.path.join(V9CFG.weights_dir, f"P{patient_id}")
    models_list = []
    for idx in range(1, V9CFG.n_ensemble + 1):
        path = os.path.join(fold_dir, f"clam_ensemble_{idx:02d}.pt")
        if not os.path.exists(path):
            print(f"  ⚠ V9 CLAM not found: {path}")
            continue
        try:
            ckpt = torch.load(path, map_location='cpu', weights_only=False)
            config = ckpt.get('config', {})
            fdim = config.get('total_dim', V9CFG.total_dim)
            hid = config.get('mil_hidden', V9CFG.mil_hidden)
            drop = config.get('mil_dropout', V9CFG.mil_dropout)
            k = config.get('clam_k', V9CFG.clam_k)
            nc = config.get('num_classes', V9CFG.num_classes)
            model = V9CLAM(fdim=fdim, nc=nc, hid=hid, drop=drop, k=k).to(DEVICE)
            model.load_state_dict(ckpt['state_dict'])
            model.eval()
            models_list.append(model)
        except RuntimeError as e:
            if 'CUDA' in str(e) or 'cuda' in str(e):
                print(f"  ⚠ CUDA error loading V9 CLAM, using CPU: {path}")
                ckpt = torch.load(path, map_location='cpu', weights_only=False)
                config = ckpt.get('config', {})
                fdim = config.get('total_dim', V9CFG.total_dim)
                hid = config.get('mil_hidden', V9CFG.mil_hidden)
                drop = config.get('mil_dropout', V9CFG.mil_dropout)
                k = config.get('clam_k', V9CFG.clam_k)
                nc = config.get('num_classes', V9CFG.num_classes)
                model = V9CLAM(fdim=fdim, nc=nc, hid=hid, drop=drop, k=k).to('cpu')
                model.load_state_dict(ckpt['state_dict'])
                model.eval()
                models_list.append(model)
            else:
                raise
    return models_list

# ── V6 prediction ──
@torch.no_grad()
def v6_predict_single(model, feats, use_mc=False):
    try:
        if use_mc:
            model.train()
        else:
            model.eval()
        feats = feats.to(next(model.parameters()).device)
        if feats.shape[0] > V6CFG.max_patches_test:
            idx = torch.randperm(feats.shape[0])[:V6CFG.max_patches_test]
            feats = feats[idx]
        with torch.amp.autocast(device_type='cuda', enabled=USE_AMP and CUDA_AVAILABLE):
            out = model(feats)
        probs = F.softmax(out['logits'].float(), dim=0)
        return probs.cpu().numpy()
    except RuntimeError as e:
        if 'CUDA' in str(e) or 'cuda' in str(e):
            print(f"⚠ CUDA error in v6_predict_single, retrying on CPU")
            model = model.cpu()
            feats = feats.cpu()
            out = model(feats)
            probs = F.softmax(out['logits'].float(), dim=0)
            return probs.cpu().numpy()
        else:
            raise

def v6_predict_ensemble(models_list, feats):
    """V6 ensemble: MC dropout + TTA + confidence-weighted geometric mean."""
    all_probs = []
    all_entropy = []
    for model in models_list:
        model_probs = []
        p = v6_predict_single(model, feats, use_mc=False)
        model_probs.append(p)
        if V6CFG.use_mc_dropout:
            for _ in range(2):
                p_mc = v6_predict_single(model, feats, use_mc=True)
                model_probs.append(p_mc)
        if V6CFG.use_tta:
            for _ in range(V6CFG.tta_rounds):
                p_tta = v6_predict_single(model, feats, use_mc=False)
                model_probs.append(p_tta)
        avg_p = np.mean(model_probs, axis=0)
        avg_p = avg_p / (avg_p.sum() + 1e-8)
        entropy = -np.sum(avg_p * np.log(avg_p + 1e-8))
        all_probs.append(avg_p)
        all_entropy.append(entropy)
    all_probs = np.array(all_probs)
    all_entropy = np.array(all_entropy)
    weights = 1.0 / (all_entropy + 0.1)
    weights = weights / weights.sum()
    log_probs = np.log(all_probs + 1e-8)
    weighted_log = np.sum(log_probs * weights[:, None], axis=0)
    geo_mean = np.exp(weighted_log)
    geo_mean = geo_mean / geo_mean.sum()
    return geo_mean

# ── V9 prediction ──
@torch.no_grad()
def v9_pred_tta(model, f, n=V9CFG.tta_rounds):
    try:
        model.eval()
        f = f.to(next(model.parameters()).device)
        ps = []
        bl,_ = model(f)
        ps.append(F.softmax(bl,-1).cpu())
        for _ in range(n-1):
            k = max(1,int(f.shape[0]*random.uniform(0.65,0.9)))
            bl,_ = model(f[torch.randperm(f.shape[0])[:k]])
            ps.append(F.softmax(bl,-1).cpu())
        return torch.stack(ps).mean(0)
    except RuntimeError as e:
        if 'CUDA' in str(e) or 'cuda' in str(e):
            model = model.cpu()
            f = f.cpu()
            model.eval()
            ps = []
            bl,_ = model(f)
            ps.append(F.softmax(bl,-1).cpu())
            for _ in range(n-1):
                k = max(1,int(f.shape[0]*random.uniform(0.65,0.9)))
                bl,_ = model(f[torch.randperm(f.shape[0])[:k]])
                ps.append(F.softmax(bl,-1).cpu())
            return torch.stack(ps).mean(0)
        else:
            raise

@torch.no_grad()
def v9_pred_mc(model, f, n=V9CFG.mc_forward):
    try:
        f = f.to(next(model.parameters()).device)
        model.eval()
        for m in model.modules():
            if isinstance(m, nn.Dropout): m.train()
        ps = []
        for _ in range(n):
            bl,_ = model(f)
            ps.append(F.softmax(bl,-1).cpu())
        model.eval()
        return torch.stack(ps).mean(0)
    except RuntimeError as e:
        if 'CUDA' in str(e) or 'cuda' in str(e):
            model = model.cpu()
            f = f.cpu()
            model.eval()
            for m in model.modules():
                if isinstance(m, nn.Dropout): m.train()
            ps = []
            for _ in range(n):
                bl,_ = model(f)
                ps.append(F.softmax(bl,-1).cpu())
            model.eval()
            return torch.stack(ps).mean(0)
        else:
            raise

def v9_predict_ensemble(models_list, feats):
    """V9 ensemble: TTA + MC dropout, averaged across models."""
    ap = []
    for m in models_list:
        f = feats
        if f.shape[0] > V9CFG.max_test_patches:
            f = feats[torch.randperm(feats.shape[0])[:V9CFG.max_test_patches]]
        pl = []
        pl.append(v9_pred_tta(m, f))
        pl.append(v9_pred_mc(m, f))
        ap.append(torch.stack(pl).mean(0))
    probs = torch.stack(ap).mean(0).squeeze(0).numpy()
    return probs

print("✓ Weight loading + prediction functions defined (with CUDA fallback)")

✓ Weight loading + prediction functions defined (with CUDA fallback)


In [10]:
# ============================================================
# CELL 10: LOPO ENSEMBLE EVALUATION
# ============================================================

print("\n" + "="*60)
print("LOPO ENSEMBLE EVALUATION")
print(f"  {N_PATIENTS} patients | V6 + V9 models")
print("="*60)

all_v6_probs = []
all_v9_probs = []
all_labels = []
all_patients_list = []
all_paths = []
fold_timing = []

for fold_idx, test_pid in enumerate(PATIENTS):
    fold_start = time.time()
    fold_num = fold_idx + 1
    test_df = df[df.patient == test_pid]
    n_test = len(test_df)

    print(f"\n  Fold {fold_num}/{N_PATIENTS} [P{test_pid}] — {n_test} test images")

    # ── Load V6 CLAM models ──
    v6_models = load_v6_models_for_fold(fold_num)
    print(f"    V6: {len(v6_models)} CLAM models loaded")

    # ── Load V9 backbone + CLAM models ──
    v9_backbone = load_v9_backbone_for_fold(test_pid)
    v9_available = v9_backbone is not None
    if v9_available:
        v9_models = load_v9_clam_for_fold(test_pid)
        v9_available = len(v9_models) > 0
        print(f"    V9: backbone + {len(v9_models)} CLAM models loaded")
    else:
        v9_models = []
        print(f"    V9: ✗ UNAVAILABLE (corrupt backbone) — V6 only for this fold")

    # ── Process each test image ──
    fold_v6_probs = []
    fold_v9_probs = []
    fold_labels = []

    for _, row in tqdm(test_df.iterrows(), total=n_test,
                       desc=f"    Fold {fold_num}", leave=False):
        try:
            img = load_image(row.path, normalizer)
        except:
            continue

        patches = extract_patches(img)
        if not patches:
            continue

        tensors = [bgr_to_tensor(p) for p in patches]

        # V6 features (1792d) and prediction
        v6_feats = extract_v6_features(tensors)
        if v6_feats.shape[0] > 0 and len(v6_models) > 0:
            v6_prob = v6_predict_ensemble(v6_models, v6_feats)
        else:
            v6_prob = np.array([1/3, 1/3, 1/3])

        # V9 features (2048d) and prediction
        if v9_available:
            v9_tensors = tensors[:V9CFG.max_patches] if len(tensors) > V9CFG.max_patches else tensors
            v9_feats = extract_v9_features(v9_tensors, v9_backbone)
            if v9_feats.shape[0] > 0:
                v9_feats_gpu = v9_feats.to(DEVICE).float()
                v9_prob = v9_predict_ensemble(v9_models, v9_feats_gpu)
            else:
                v9_prob = np.array([1/3, 1/3, 1/3])
        else:
            # V9 unavailable — use uniform (handled in ensemble analysis)
            v9_prob = np.array([1/3, 1/3, 1/3])

        fold_v6_probs.append(v6_prob)
        fold_v9_probs.append(v9_prob)
        fold_labels.append(int(row.target))
        all_v6_probs.append(v6_prob)
        all_v9_probs.append(v9_prob)
        all_labels.append(int(row.target))
        all_patients_list.append(int(row.patient))
        all_paths.append(row.path)

    # ── Fold summary ──
    if fold_labels:
        fold_v6_preds = np.argmax(fold_v6_probs, axis=1)
        fold_v9_preds = np.argmax(fold_v9_probs, axis=1)
        fold_labels_np = np.array(fold_labels)
        v6_acc = accuracy_score(fold_labels_np, fold_v6_preds)
        v9_acc = accuracy_score(fold_labels_np, fold_v9_preds)
        # Simple ensemble (equal weight average)
        fold_ens_probs = 0.5 * np.array(fold_v6_probs) + 0.5 * np.array(fold_v9_probs)
        ens_acc = accuracy_score(fold_labels_np, np.argmax(fold_ens_probs, axis=1))
        fold_time = time.time() - fold_start
        v9_tag = f"V9={v9_acc:.1%}" if v9_available else "V9=N/A"
        ens_tag = f"Ens={ens_acc:.1%}" if v9_available else "Ens=V6only"
        print(f"    V6={v6_acc:.1%}  {v9_tag}  {ens_tag} | {fold_time/60:.1f}m")
        fold_timing.append({'patient': test_pid, 'v6_acc': v6_acc,
                           'v9_acc': v9_acc if v9_available else None,
                           'ens_acc': ens_acc, 'v9_available': v9_available,
                           'time_min': fold_time/60})

    # ── Cleanup ──
    for m in v6_models: del m
    for m in v9_models: del m
    del v6_models, v9_models
    # Reset V9 backbone
    dino_v9_model.load_state_dict(original_v9_sd)
    torch.cuda.empty_cache(); gc.collect()

print(f"\n{'='*60}")
print(f"✓ LOPO evaluation complete — {len(all_labels)} images processed")

# Convert to arrays
all_v6_probs = np.array(all_v6_probs)
all_v9_probs = np.array(all_v9_probs)
all_labels = np.array(all_labels)
all_patients_arr = np.array(all_patients_list)


LOPO ENSEMBLE EVALUATION
  14 patients | V6 + V9 models

  Fold 1/14 [P1] — 491 test images
    V6: 3 CLAM models loaded
    V9: backbone + 3 CLAM models loaded


    V6=86.8%  V9=83.1%  Ens=88.2% | 11.0m

  Fold 2/14 [P2] — 302 test images
    V6: 3 CLAM models loaded
    V9: backbone + 3 CLAM models loaded


    V6=93.4%  V9=90.1%  Ens=94.0% | 6.9m

  Fold 3/14 [P3] — 172 test images
    V6: 3 CLAM models loaded
    V9: backbone + 3 CLAM models loaded


    V6=86.6%  V9=80.2%  Ens=86.6% | 4.0m

  Fold 4/14 [P4] — 234 test images
    V6: 3 CLAM models loaded
    V9: backbone + 3 CLAM models loaded


    V6=74.8%  V9=85.5%  Ens=81.6% | 5.4m

  Fold 5/14 [P5] — 175 test images
    V6: 3 CLAM models loaded
    V9: backbone + 3 CLAM models loaded


...

    V6=100.0%  V9=100.0%  Ens=100.0% | 1.3m

  Fold 11/14 [P11] — 46 test images
    V6: 3 CLAM models loaded
    V9: backbone + 3 CLAM models loaded


    V6=100.0%  V9=100.0%  Ens=100.0% | 1.2m

  Fold 12/14 [P12] — 19 test images
    V6: 3 CLAM models loaded
    V9: backbone + 3 CLAM models loaded


    V6=94.7%  V9=100.0%  Ens=94.7% | 0.7m

  Fold 13/14 [P13] — 3 test images
    V6: 3 CLAM models loaded
    V9: backbone + 3 CLAM models loaded


    V6=100.0%  V9=100.0%  Ens=100.0% | 0.2m

  Fold 14/14 [P14] — 4 test images
    V6: 3 CLAM models loaded
    V9: backbone + 3 CLAM models loaded


    V6=50.0%  V9=75.0%  Ens=50.0% | 0.2m

✓ LOPO evaluation complete — 1713 images processed


In [11]:
# ============================================================
# CELL 11: ENSEMBLE COMBINATION & ANALYSIS
# ============================================================

print("\n" + "="*60)
print("ENSEMBLE ANALYSIS")
print("="*60)

# ── Helper functions ──
def opt_thresholds(probs, labels, n_classes=3, n_restarts=50, recall_priority=1.5):
    """ Optimise per-class thresholds to maximise balanced accuracy + HGC recall."""
    def obj(th):
        adj = probs * th
        pr = adj.argmax(1)
        ba = balanced_accuracy_score(labels, pr)
        hm = labels == 0
        hr = (pr[hm] == 0).mean() if hm.sum() > 0 else 0
        pen = sum(max(0, 0.5 - (pr[labels == c] == c).mean()) * 0.5
                  for c in range(n_classes) if (labels == c).sum() > 0)
        return -(ba + recall_priority * hr - pen)
    best = None; bs = float('inf')
    for _ in range(n_restarts):
        x0 = np.random.uniform(0.6, 1.6, n_classes)
        x0 = x0 / x0.sum() * n_classes
        r = minimize(obj, x0, method='Nelder-Mead', options={'maxiter': 1000})
        if r.fun < bs: bs = r.fun; best = r.x
    return best / best.sum() * n_classes

def patient_vote(preds, labels, patients):
    pd2 = defaultdict(lambda: {'p': [], 'l': []})
    for p, l, pid in zip(preds, labels, patients):
        pd2[pid]['p'].append(p); pd2[pid]['l'].append(l)
    pp, pl, pi = [], [], []
    for pid in sorted(pd2):
        pp.append(Counter(pd2[pid]['p']).most_common(1)[0][0])
        pl.append(Counter(pd2[pid]['l']).most_common(1)[0][0])
        pi.append(pid)
    return pp, pl, pi

def evaluate_probs(probs, labels, tag=""):
    """Evaluate a set of probability predictions."""
    preds = probs.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    bal = balanced_accuracy_score(labels, preds)
    cm = confusion_matrix(labels, preds, labels=[0,1,2])
    recalls = [cm[i,i]/max(cm[i].sum(),1) for i in range(3)]
    print(f"\n{'─'*50}")
    print(f"  {tag}")
    print(f"  Accuracy:          {acc:.1%}")
    print(f"  Balanced Accuracy: {bal:.1%}")
    for i,c in enumerate(CFG.class_names):
        print(f"    {c}: {recalls[i]:.1%} ({cm[i,i]}/{cm[i].sum()})")
    return {'acc': acc, 'bal': bal, 'recalls': recalls, 'cm': cm, 'preds': preds}

# ── 1. Individual model results (raw) ──
print("\n" + "═"*60)
print("  INDIVIDUAL MODEL RESULTS (RAW)")
print("═"*60)
v6_raw = evaluate_probs(all_v6_probs, all_labels, "V6 (DINOv2-B + DenseNet, 1792d)")
v9_raw = evaluate_probs(all_v9_probs, all_labels, "V9 (DINOv2-L[ft] + DenseNet, 2048d)")

# ── 2. Individual model results (threshold optimized) ──
print("\n" + "═"*60)
print("  INDIVIDUAL MODEL RESULTS (THRESHOLD OPTIMIZED)")
print("═"*60)

v6_th = opt_thresholds(all_v6_probs, all_labels)
v6_adj = all_v6_probs * v6_th
v6_opt = evaluate_probs(v6_adj, all_labels, f"V6 Optimized (th={[f'{t:.2f}' for t in v6_th]})")

v9_th = opt_thresholds(all_v9_probs, all_labels)
v9_adj = all_v9_probs * v9_th
v9_opt = evaluate_probs(v9_adj, all_labels, f"V9 Optimized (th={[f'{t:.2f}' for t in v9_th]})")

# ── 3. Ensemble: Equal weights ──
print("\n" + "═"*60)
print("  ENSEMBLE RESULTS")
print("═"*60)

ens_equal = 0.5 * all_v6_probs + 0.5 * all_v9_probs
ens_equal_res = evaluate_probs(ens_equal, all_labels, "Ensemble (equal weights: 0.5/0.5)")

# ── 4. Ensemble: Optimal weight search ──
def search_optimal_weight(v6_probs, v9_probs, labels):
    """Search for optimal mixing weight between V6 and V9."""
    best_w = 0.5
    best_score = -1
    results = []
    for w in np.arange(0.0, 1.05, 0.05):
        mixed = w * v6_probs + (1 - w) * v9_probs
        preds = mixed.argmax(axis=1)
        bal = balanced_accuracy_score(labels, preds)
        acc = accuracy_score(labels, preds)
        hgc_mask = labels == 0
        hgc_rec = (preds[hgc_mask] == 0).mean() if hgc_mask.sum() > 0 else 0
        score = bal + 0.5 * hgc_rec  # Weighted score
        results.append({'w_v6': w, 'acc': acc, 'bal': bal, 'hgc_rec': hgc_rec, 'score': score})
        if score > best_score:
            best_score = score
            best_w = w
    return best_w, pd.DataFrame(results)

best_w, weight_search_df = search_optimal_weight(all_v6_probs, all_v9_probs, all_labels)
print(f"\n  Optimal weight: V6={best_w:.2f}, V9={1-best_w:.2f}")

ens_optimal = best_w * all_v6_probs + (1 - best_w) * all_v9_probs
ens_opt_res = evaluate_probs(ens_optimal, all_labels,
                              f"Ensemble (optimal weights: V6={best_w:.2f}/V9={1-best_w:.2f})")

# ── 5. Ensemble with threshold optimization ──
ens_th = opt_thresholds(ens_optimal, all_labels)
ens_adj = ens_optimal * ens_th
ens_th_res = evaluate_probs(ens_adj, all_labels,
                             f"Ensemble (optimal+threshold: th={[f'{t:.2f}' for t in ens_th]})")

# ── 6. Patient-level voting ──
print("\n" + "═"*60)
print("  PATIENT-LEVEL MAJORITY VOTE")
print("═"*60)

for tag, preds in [("V6 raw", v6_raw['preds']),
                   ("V9 raw", v9_raw['preds']),
                   ("Ensemble (equal)", ens_equal_res['preds']),
                   ("Ensemble (optimal)", ens_opt_res['preds']),
                   ("Ensemble (opt+thresh)", ens_th_res['preds'])]:
    pp, pl, pi = patient_vote(preds.tolist(), all_labels.tolist(), all_patients_list)
    pa = accuracy_score(pl, pp)
    pb = balanced_accuracy_score(pl, pp)
    print(f"  {tag:30s}: Acc={pa:.1%}  Bal={pb:.1%}")

# ── 7. Cancer detection (binary) ──
print("\n" + "═"*60)
print("  CANCER DETECTION (binary: HGC+LGC vs Normal)")
print("═"*60)

for tag, preds in [("V6 raw", v6_raw['preds']),
                   ("V9 raw", v9_raw['preds']),
                   ("Ensemble (optimal)", ens_opt_res['preds']),
                   ("Ensemble (opt+thresh)", ens_th_res['preds'])]:
    bl = [0 if l < 2 else 1 for l in all_labels]
    bp = [0 if p < 2 else 1 for p in preds]
    ca = accuracy_score(bl, bp)
    ccm = confusion_matrix(bl, bp, labels=[0,1])
    sens = ccm[0,0]/max(ccm[0].sum(),1)
    spec = ccm[1,1]/max(ccm[1].sum(),1)
    print(f"  {tag:30s}: Acc={ca:.1%}  Sens={sens:.1%}  Spec={spec:.1%}")

# ── 8. Comprehensive comparison table ──
print("\n" + "═"*60)
print("  COMPREHENSIVE COMPARISON")
print("═"*60)
comparison = pd.DataFrame([
    {'Model': 'V6 Raw', 'Accuracy': v6_raw['acc'], 'Bal Acc': v6_raw['bal'],
     'HGC Recall': v6_raw['recalls'][0], 'LGC Recall': v6_raw['recalls'][1],
     'Normal Recall': v6_raw['recalls'][2]},
    {'Model': 'V6 Optimized', 'Accuracy': v6_opt['acc'], 'Bal Acc': v6_opt['bal'],
     'HGC Recall': v6_opt['recalls'][0], 'LGC Recall': v6_opt['recalls'][1],
     'Normal Recall': v6_opt['recalls'][2]},
    {'Model': 'V9 Raw', 'Accuracy': v9_raw['acc'], 'Bal Acc': v9_raw['bal'],
     'HGC Recall': v9_raw['recalls'][0], 'LGC Recall': v9_raw['recalls'][1],
     'Normal Recall': v9_raw['recalls'][2]},
    {'Model': 'V9 Optimized', 'Accuracy': v9_opt['acc'], 'Bal Acc': v9_opt['bal'],
     'HGC Recall': v9_opt['recalls'][0], 'LGC Recall': v9_opt['recalls'][1],
     'Normal Recall': v9_opt['recalls'][2]},
    {'Model': 'Ensemble Equal', 'Accuracy': ens_equal_res['acc'], 'Bal Acc': ens_equal_res['bal'],
     'HGC Recall': ens_equal_res['recalls'][0], 'LGC Recall': ens_equal_res['recalls'][1],
     'Normal Recall': ens_equal_res['recalls'][2]},
    {'Model': f'Ensemble Opt (w={best_w:.2f})', 'Accuracy': ens_opt_res['acc'],
     'Bal Acc': ens_opt_res['bal'], 'HGC Recall': ens_opt_res['recalls'][0],
     'LGC Recall': ens_opt_res['recalls'][1], 'Normal Recall': ens_opt_res['recalls'][2]},
    {'Model': 'Ensemble Opt+Thresh', 'Accuracy': ens_th_res['acc'], 'Bal Acc': ens_th_res['bal'],
     'HGC Recall': ens_th_res['recalls'][0], 'LGC Recall': ens_th_res['recalls'][1],
     'Normal Recall': ens_th_res['recalls'][2]},
])
for col in ['Accuracy','Bal Acc','HGC Recall','LGC Recall','Normal Recall']:
    comparison[col] = comparison[col].apply(lambda x: f"{x:.1%}")
print(comparison.to_string(index=False))

# Full classification report for best ensemble
print("\n" + "─"*60)
print("  CLASSIFICATION REPORT (Best Ensemble)")
print("─"*60)
best_ens_preds = ens_th_res['preds']
print(classification_report(all_labels, best_ens_preds,
                           target_names=CFG.class_names, digits=4))
try:
    f1w = f1_score(all_labels, best_ens_preds, average='weighted')
    f1m = f1_score(all_labels, best_ens_preds, average='macro')
    kap = cohen_kappa_score(all_labels, best_ens_preds)
    print(f"F1(weighted)={f1w:.4f} | F1(macro)={f1m:.4f} | Cohen's κ={kap:.4f}")
except: pass


ENSEMBLE ANALYSIS

════════════════════════════════════════════════════════════
  INDIVIDUAL MODEL RESULTS (RAW)
════════════════════════════════════════════════════════════

──────────────────────────────────────────────────
  V6 (DINOv2-B + DenseNet, 1792d)
  Accuracy:          87.9%
  Balanced Accuracy: 87.3%
    HGC: 79.1% (371/469)
    LGC: 87.2% (564/647)
    Normal: 95.6% (571/597)

──────────────────────────────────────────────────
  V9 (DINOv2-L[ft] + DenseNet, 2048d)
  Accuracy:          87.1%
  Balanced Accuracy: 86.2%
    HGC: 74.8% (351/469)
    LGC: 89.3% (578/647)
    Normal: 94.3% (563/597)

════════════════════════════════════════════════════════════
  INDIVIDUAL MODEL RESULTS (THRESHOLD OPTIMIZED)
════════════════════════════════════════════════════════════

──────────────────────────────────────────────────
  V6 Optimized (th=['2.60', '0.25', '0.15'])
  Accuracy:          82.3%
  Balanced Accuracy: 83.1%
    HGC: 88.9% (417/469)
    LGC: 70.9% (459/647)
    Normal: 

In [12]:
# ============================================================
# CELL 12: VISUALIZATION
# ============================================================

fig = plt.figure(figsize=(24, 16))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# ── Confusion Matrices ──
cms = [
    (v6_raw['cm'], "V6 Raw"),
    (v9_raw['cm'], "V9 Raw"),
    (ens_th_res['cm'], "Ensemble (Opt+Thresh)"),
]
for i, (cm, title) in enumerate(cms):
    ax = fig.add_subplot(gs[0, i])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CFG.class_names, yticklabels=CFG.class_names, ax=ax)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('True' if i == 0 else '')
    ax.set_xlabel('Predicted')

# ── Per-patient accuracy comparison ──
ax4 = fig.add_subplot(gs[1, 0])
if fold_timing:
    timing_df = pd.DataFrame(fold_timing)
    x = np.arange(len(timing_df))
    w = 0.25
    ax4.bar(x - w, timing_df['v6_acc'], w, label='V6', color='#4FC3F7')
    ax4.bar(x, timing_df['v9_acc'], w, label='V9', color='#FF8A65')
    ax4.bar(x + w, timing_df['ens_acc'], w, label='Ensemble', color='#81C784')
    ax4.set_xticks(x)
    ax4.set_xticklabels([f"P{p}" for p in timing_df['patient']], rotation=45)
    ax4.set_ylabel('Accuracy')
    ax4.set_title('Per-Patient Accuracy', fontweight='bold')
    ax4.legend()
    ax4.set_ylim(0, 1.05)

# ── Weight sensitivity ──
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(weight_search_df['w_v6'], weight_search_df['bal'], 'b-o', label='Balanced Acc', markersize=4)
ax5.plot(weight_search_df['w_v6'], weight_search_df['hgc_rec'], 'r--s', label='HGC Recall', markersize=4)
ax5.plot(weight_search_df['w_v6'], weight_search_df['acc'], 'g-^', label='Accuracy', markersize=4)
ax5.axvline(best_w, color='gray', linestyle=':', alpha=0.7, label=f'Optimal w={best_w:.2f}')
ax5.set_xlabel('V6 Weight (V9 = 1 - V6)')
ax5.set_ylabel('Metric')
ax5.set_title('Ensemble Weight Sensitivity', fontweight='bold')
ax5.legend(fontsize=8)
ax5.set_ylim(0, 1.05)

# ── Per-class recall comparison ──
ax6 = fig.add_subplot(gs[1, 2])
models_labels = ['V6 Raw', 'V9 Raw', 'Ens Equal', f'Ens Opt', 'Ens+Thresh']
hgc_recs = [v6_raw['recalls'][0], v9_raw['recalls'][0], ens_equal_res['recalls'][0],
            ens_opt_res['recalls'][0], ens_th_res['recalls'][0]]
lgc_recs = [v6_raw['recalls'][1], v9_raw['recalls'][1], ens_equal_res['recalls'][1],
            ens_opt_res['recalls'][1], ens_th_res['recalls'][1]]
nrm_recs = [v6_raw['recalls'][2], v9_raw['recalls'][2], ens_equal_res['recalls'][2],
            ens_opt_res['recalls'][2], ens_th_res['recalls'][2]]
x = np.arange(len(models_labels))
w = 0.25
ax6.bar(x - w, hgc_recs, w, label='HGC', color='#EF5350')
ax6.bar(x, lgc_recs, w, label='LGC', color='#FFA726')
ax6.bar(x + w, nrm_recs, w, label='Normal', color='#66BB6A')
ax6.set_xticks(x)
ax6.set_xticklabels(models_labels, rotation=30, ha='right')
ax6.set_ylabel('Recall')
ax6.set_title('Per-Class Recall Comparison', fontweight='bold')
ax6.legend()
ax6.set_ylim(0, 1.15)

plt.suptitle('Bladder Cancer Classification — V6 + V9 Ensemble', fontsize=16, fontweight='bold', y=0.98)
plt.savefig(os.path.join(CFG.results_dir, 'ensemble_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Visualization saved to {CFG.results_dir}/ensemble_analysis.png")

✓ Visualization saved to ensemble_results/ensemble_analysis.png


In [14]:
# ============================================================
# CELL 13: SAVE RESULTS
# ============================================================

# Convert numpy types to native Python types for JSON serialization
def convert_to_native(obj):
    """Recursively convert numpy types to native Python types."""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_native(item) for item in obj]
    return obj

# Prepare fold_timing with native types
fold_timing_native = convert_to_native(fold_timing)

results = {
    'v6_probs': all_v6_probs.tolist(),
    'v9_probs': all_v9_probs.tolist(),
    'labels': all_labels.tolist(),
    'patients': [int(p) for p in all_patients_list],
    'paths': all_paths,
    'optimal_v6_weight': float(best_w),
    'v6_thresholds': v6_th.tolist(),
    'v9_thresholds': v9_th.tolist(),
    'ensemble_thresholds': ens_th.tolist(),
    'metrics': {
        'v6_raw': {'acc': float(v6_raw['acc']), 'bal': float(v6_raw['bal']), 
                   'recalls': [float(r) for r in v6_raw['recalls']]},
        'v9_raw': {'acc': float(v9_raw['acc']), 'bal': float(v9_raw['bal']), 
                   'recalls': [float(r) for r in v9_raw['recalls']]},
        'v6_opt': {'acc': float(v6_opt['acc']), 'bal': float(v6_opt['bal']), 
                   'recalls': [float(r) for r in v6_opt['recalls']]},
        'v9_opt': {'acc': float(v9_opt['acc']), 'bal': float(v9_opt['bal']), 
                   'recalls': [float(r) for r in v9_opt['recalls']]},
        'ens_equal': {'acc': float(ens_equal_res['acc']), 'bal': float(ens_equal_res['bal']),
                      'recalls': [float(r) for r in ens_equal_res['recalls']]},
        'ens_optimal': {'acc': float(ens_opt_res['acc']), 'bal': float(ens_opt_res['bal']),
                        'recalls': [float(r) for r in ens_opt_res['recalls']]},
        'ens_opt_thresh': {'acc': float(ens_th_res['acc']), 'bal': float(ens_th_res['bal']),
                           'recalls': [float(r) for r in ens_th_res['recalls']]},
    },
    'fold_timing': fold_timing_native,
    'weight_search': convert_to_native(weight_search_df.to_dict('records')),
}

with open(os.path.join(CFG.results_dir, 'ensemble_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

# Save raw probabilities as numpy
np.save(os.path.join(CFG.results_dir, 'v6_probs.npy'), all_v6_probs)
np.save(os.path.join(CFG.results_dir, 'v9_probs.npy'), all_v9_probs)
np.save(os.path.join(CFG.results_dir, 'labels.npy'), all_labels)

print(f"\n✓ Results saved to {CFG.results_dir}/")
print(f"  ensemble_results.json — all metrics and config")
print(f"  v6_probs.npy, v9_probs.npy, labels.npy — raw data")
print(f"  ensemble_analysis.png — visualization")


✓ Results saved to ensemble_results/
  ensemble_results.json — all metrics and config
  v6_probs.npy, v9_probs.npy, labels.npy — raw data
  ensemble_analysis.png — visualization
